# Stage 06 - Validate Authoritative Ontology Bindings

Confirm the required Stage 03 Lakehouse tables and Stage 04 Eventhouse bindings. This notebook never creates substitute data.

In [ ]:
# This notebook validates authoritative products; it never generates substitute ontology data.
RELEASE_ID = '2026.11.03'
print(f'Validating ontology bindings for shared release {RELEASE_ID}')

In [ ]:
import csv
from pathlib import Path
from zipfile import ZipFile

package = Path('/lakehouse/default/Files/fabric-demos/06-ai-data-agent/Ontology/mda_test_ontology.iq')
with ZipFile(package) as archive:
    bindings = list(csv.DictReader(archive.read('binding/binding_entity_types.csv').decode().splitlines()))
    relationships = list(csv.DictReader(archive.read('binding/binding_relationship_types.csv').decode().splitlines()))
required_columns = {}
for binding in bindings:
    if binding['SourceType'] == 'LakehouseTable':
        required_columns.setdefault(binding['SourceTableName'], set()).add(binding['BindingSourceColumnName'])
for relationship in relationships:
    columns = relationship['SourceKeyColumnNames'].split(',') + relationship['TargetKeyColumnNames'].split(',')
    required_columns.setdefault(relationship['SourceTableName'], set()).update(columns)
for table, columns in required_columns.items():
    frame = spark.table(table)
    missing = columns - set(frame.columns)
    if missing or frame.limit(1).count() == 0:
        raise RuntimeError(f'Invalid Lakehouse binding {table}: missing columns {sorted(missing)} or empty table')
print(f'Lakehouse binding columns verified: {sorted(required_columns)}')

In [ ]:
import json
from datetime import datetime, timezone
import requests
import sempy.fabric as fabric
from notebookutils import mssparkutils

workspace_id = fabric.get_workspace_id()
items = fabric.list_items()
database_items = items[(items['Type'] == 'KQLDatabase') & (items['Display Name'] == 'kqldb_mda_test')]
if len(database_items) != 1:
    raise RuntimeError('Expected one kqldb_mda_test database')
database_id = str(database_items.iloc[0].Id)
client = fabric.FabricRestClient()
database = client.get(f'/v1/workspaces/{workspace_id}/kqlDatabases/{database_id}').json()
query_uri = database['properties']['queryServiceUri']
access_token = mssparkutils.credentials.getToken('https://kusto.kusto.windows.net')

def normalize(record):
    result = dict(record)
    for name, value in result.items():
        if name.endswith('_time_utc'):
            instant = value if isinstance(value, datetime) else datetime.fromisoformat(value.replace('Z', '+00:00'))
            result[name] = instant.replace(tzinfo=timezone.utc).isoformat(timespec='microseconds')
    return result

for entity, table, key in [('Observation', 'OntologySensorObservation', 'observation_id'), ('CommandEvent', 'OntologyCommandEvent', 'command_event_id')]:
    entity_bindings = [binding for binding in bindings if binding['EntityTypeName'] == entity]
    static_binding = next(binding for binding in entity_bindings if binding['DataBindingType'] == 'NonTimeSeries')
    if entity == 'Observation':
        expected_frame = spark.table(static_binding['SourceTableName']).selectExpr('event_id as observation_id', 'track_id', 'system_instance_id', 'observation_type', 'event_time_utc as observation_time_utc', 'quality_score')
    else:
        expected_frame = spark.table(static_binding['SourceTableName']).selectExpr('event_id as command_event_id', 'track_id', 'system_instance_id', 'action as action_name', 'status as action_status', 'event_time_utc', 'cast(processing_delay_milliseconds as bigint) as processing_delay_ms')
    expected = [normalize(row.asDict()) for row in expected_frame.collect()]
    keys = [row[key] for row in expected]
    if not keys or len(set(keys)) != len(keys):
        raise RuntimeError(f'{entity}: empty or nonunique identities')
    query = table + ' | where ' + key + ' in (' + ', '.join(json.dumps(value) for value in keys) + ')'
    response = requests.post(query_uri + '/v2/rest/query', headers={'Authorization': 'Bearer ' + access_token}, json={'db': database_id, 'csl': query}, timeout=120)
    response.raise_for_status()
    frames = response.json()
    if any(frame.get('HasErrors') for frame in frames):
        raise RuntimeError(f'{entity}: Eventhouse query failed')
    result = next(frame for frame in frames if frame.get('TableKind') == 'PrimaryResult')
    columns = [column['ColumnName'] for column in result['Columns']]
    actual = [normalize(dict(zip(columns, row))) for row in result['Rows']]
    if sorted(actual, key=lambda row: row[key]) != sorted(expected, key=lambda row: row[key]):
        raise RuntimeError(f'{entity}: Eventhouse history differs from Lakehouse facts; run provision-demo-06-ontology')
    print(f'{entity}: verified {len(actual)} exact history records')
print('No instance or event data was generated by this notebook.')